### Imports and Loading Model

In [1]:
# Import
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
import chromadb

C:\Users\Raghu Sanivarapu\venvs\simple-rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4327.27it/s]

### Knowledge Base

In [3]:
# 10 IT help desk articles, each one starting with a "## " heading
knowledge_base = """
## VPN Timeout

If the VPN times out when you work from home, check your internet connection and restart the VPN app.
The VPN needs UDP ports 500 and 4500 open. If it still fails, contact the network team.

## MFA Setup

If Microsoft Authenticator won't finish setting up MFA, first set your phone's date and time to automatic.
Then reinstall the app and scan the QR code again.

## Slow Laptop

If your laptop is slow, let Windows updates finish, close unused browser tabs and restart.
Keep at least 15% of the disk free.

## Salesforce Access Denied

After a role change, Salesforce can show Access Denied for up to 30 minutes while access updates.
Wait 30 minutes, then sign out and back in. If it still fails, ask your manager to approve the new role.

## Office WiFi

Connect to the CorpWiFi-5G network. If the WiFi keeps dropping, move closer to an access point
or restart your laptop's WiFi. Building C has older access points, due for an upgrade.

## Password Reset

A new password needs at least 12 characters, with an uppercase letter, a lowercase letter, a number and a special character.
If the reset page shows error 0x80070005, try another browser, or call the help desk for a manual reset.

## Printer Not Found

To add the HP-Floor3-Color printer, connect to the office network (not guest WiFi) and add the printer
by its IP address, 10.50.3.25.

## Email Attachment Limit

Email attachments can be up to 25 MB to people outside the company and 50 MB inside it.
For bigger files, share a OneDrive link instead.

## Database Timeout

If the production database times out at peak hours (9 to 11 AM and 2 to 4 PM), look for slow queries
and check the connection pool settings.

## Teams Audio

If you hear nothing in Teams meetings, pick the right speaker in Teams settings, make sure Windows isn't muted,
and restart Teams.
"""

# One article per heading: the heading becomes its "title", the rest its text
articles = MarkdownHeaderTextSplitter([("##", "title")]).split_text(knowledge_base)
print("Articles:", len(articles))

Articles: 10


### Chunking

In [4]:
# Cut each article into chunks of up to 150 characters: a longer article splits at its line breaks, a shorter one stays whole
splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=0)
chunks = splitter.split_documents(articles)   # each chunk keeps its article's title

print("Chunks:", len(chunks))
for i, chunk in enumerate(chunks):
    print(f"C{i+1} [{chunk.metadata['title']}] {chunk.page_content}")

Chunks: 15
C1 [VPN Timeout] If the VPN times out when you work from home, check your internet connection and restart the VPN app.
C2 [VPN Timeout] The VPN needs UDP ports 500 and 4500 open. If it still fails, contact the network team.
C3 [MFA Setup] If Microsoft Authenticator won't finish setting up MFA, first set your phone's date and time to automatic.
C4 [MFA Setup] Then reinstall the app and scan the QR code again.
C5 [Slow Laptop] If your laptop is slow, let Windows updates finish, close unused browser tabs and restart.
Keep at least 15% of the disk free.
C6 [Salesforce Access Denied] After a role change, Salesforce can show Access Denied for up to 30 minutes while access updates.
C7 [Salesforce Access Denied] Wait 30 minutes, then sign out and back in. If it still fails, ask your manager to approve the new role.
C8 [Office WiFi] Connect to the CorpWiFi-5G network. If the WiFi keeps dropping, move closer to an access point
C9 [Office WiFi] or restart your laptop's WiFi. Building C

### Indexing Data

In [5]:
texts = [chunk.page_content for chunk in chunks]
titles = [chunk.metadata for chunk in chunks]
ids = [f"C{i+1}" for i in range(len(chunks))]

# Setup Chroma client and collection, in memory
chroma_client = chromadb.Client()
# Drop existing collection if it already exists
try:
    chroma_client.delete_collection("it_kb")
except Exception:
    pass

# Now create fresh: "cosine" makes the distance 1 - cosine similarity
collection = chroma_client.create_collection("it_kb", configuration={"hnsw": {"space": "cosine"}})

# Add chunks with embeddings
embeddings = model.encode(texts).tolist()
collection.add(ids=ids, documents=texts, embeddings=embeddings, metadatas=titles)

print("Indexed chunks:", collection.count())

Indexed chunks: 15


### Basic RAG Query

In [6]:
query = "Which ports does the VPN need open?"
query_emb = model.encode([query]).tolist()

results = collection.query(query_embeddings=query_emb, n_results=3)

print("Question:", query)
print("Basic RAG Results:")
for chunk_id, text, meta, distance in zip(results["ids"][0], results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"{chunk_id} [{meta['title']}] similarity={1 - distance:.4f}: {text}")

Question: Which ports does the VPN need open?
Basic RAG Results:
C2 [VPN Timeout] similarity=0.5991: The VPN needs UDP ports 500 and 4500 open. If it still fails, contact the network team.
C1 [VPN Timeout] similarity=0.3794: If the VPN times out when you work from home, check your internet connection and restart the VPN app.
C9 [Office WiFi] similarity=0.1107: or restart your laptop's WiFi. Building C has older access points, due for an upgrade.
